# 05 — Comparaison LLM et Prompts
**RQ4** : Quel prompt et quel LLM produisent les réponses les plus fidèles et pertinentes ?

3 templates × 2 LLMs = 6 configurations.

In [ ]:
import sys, os, pandas as pd, numpy as np
sys.path.insert(0, os.path.abspath(".."))
sys.path.insert(0, os.path.abspath("../src"))

from config import load_config, LLMConfig
from retriever import retrieve_documents
from llm_chain import generate_answer, build_prompt
from evaluation_judge import create_judge
from deepeval.test_case import LLMTestCase
from deepeval.metrics import FaithfulnessMetric, AnswerRelevancyMetric
from langchain_core.prompts import PromptTemplate

from notebooks.lib.reporter import load_benchmark
from notebooks.lib.plotter import barplot, boxplot
from experiments.registry import ExperimentLog

In [ ]:
benchmark = load_benchmark()
cfg = load_config()
judge = create_judge(cfg.evaluation)

TEMPLATES = {
    "direct": PromptTemplate.from_template(
        "Contexte : {context}\n\nQuestion : {question}\n\nRéponse :"
    ),
    "pédagogique": PromptTemplate.from_template(
        "Tu es un assistant Master SIM.\n\nContexte : {context}\n\n"
        "Question d'étudiant : {question}\n\nRÉPONSE PÉDAGOGIQUE :"
    ),
    "structuré": PromptTemplate.from_template(
        "[SYSTÈME] Assistant académique spécialisé dans les règlements SIM.\n"
        "[CONTEXTE] {context}\n"
        "[QUESTION] {question}\n"
        "[RÈGLE] Réponds uniquement à partir du contexte.\n"
        "[RÉPONSE]"
    ),
}

LLM_MODELS = ["qwen2.5:3b", "mistral:7b"]

log = ExperimentLog(name="llm_prompt_comparison")
log.set_params(templates=list(TEMPLATES.keys()), llms=LLM_MODELS)

In [ ]:
for llm_name in LLM_MODELS:
    llm_cfg = LLMConfig(provider="ollama", model=llm_name, temperature=0.2)

    for tmpl_name, tmpl in TEMPLATES.items():
        print(f"\n=== LLM: {llm_name}, Prompt: {tmpl_name} ===")

        for idx, row in benchmark.iterrows():
            docs, scores = retrieve_documents(row['Question'], cfg.retrieval)
            if not docs:
                continue

            context = "\n".join([d.page_content for d in docs])
            prompt_text = tmpl.format(context=context, question=row['Question'])

            from llm_factory import create_llm, llm_invoke
            llm = create_llm(llm_cfg)
            answer = llm_invoke(llm, prompt_text)

            contexts = [d.page_content for d in docs]
            tc = LLMTestCase(
                input=row['Question'], actual_output=answer,
                expected_output=row['Ground_Truth'],
                retrieval_context=contexts,
            )

            fm = FaithfulnessMetric(threshold=0.75, model=judge, include_reason=True)
            ar = AnswerRelevancyMetric(threshold=0.75, model=judge, include_reason=True)

            try:
                fm.measure(tc); f_score = round(fm.score, 4)
            except: f_score = 0.0
            try:
                ar.measure(tc); a_score = round(ar.score, 4)
            except: a_score = 0.0

            log.record(ID=row['ID'], llm=llm_name, prompt=tmpl_name,
                       Faithfulness=f_score, Answer_Relevancy=a_score)

log.save_all()
log.append_to_global_log()
print("\nExpérience terminée.")

In [ ]:
df = pd.DataFrame(log.results)
avg = df.groupby(['llm', 'prompt'])[['Faithfulness', 'Answer_Relevancy']].mean().round(4)
print("=== Résultats moyens par LLM x Prompt ===")
print(avg)

barplot(df, x='prompt', y='Faithfulness', hue='llm',
        title="Faithfulness par prompt et LLM",
        filename='llm_prompt_faithfulness.png')